# Supplementary Table: Discrimination Statistics for All Features

Assembles one table from the saved outputs of the analysis notebooks (no statistics are recomputed here):
- `table_auc_results.csv` (`composite_figures.ipynb`): lobar composites, asymmetry indices, tract-level FA/MD.
- `table_burden_auc.csv`, `table_hippocampus_auc.csv`, `table_burden_nonlesional.csv`
  (`abnormality_burden_summary.ipynb`): whole-brain burden, hippocampal measures, positive control, non-lesional burden.

AUC is shown as max(AUC, 1 - AUC) with the group having higher values named. q is Benjamini-Hochberg within each
family: lobar composites (7), asymmetry indices (7), tract-level FA and MD pooled (18), burden (6), hippocampal (2).

Output: `../outputs/table_supp_discrimination_all.csv`.

In [1]:
import pandas as pd
import numpy as np
out_dir = '../outputs'
main = pd.read_csv(f'{out_dir}/table_auc_results.csv')
burden = pd.read_csv(f'{out_dir}/table_burden_auc.csv')
hip = pd.read_csv(f'{out_dir}/table_hippocampus_auc.csv')
nl = pd.read_csv(f'{out_dir}/table_burden_nonlesional.csv')
assert {'p', 'q'} <= set(main.columns), 'run composite_figures.ipynb first (p column missing)'

TRACT_NAMES = {'Cingulum': 'Cingulum', 'Fornix': 'Fornix', 'Uncinate': 'Uncinate fasciculus',
               'Arcuate': 'Arcuate fasciculus', 'IFOF': 'Inferior fronto-occipital fasciculus',
               'CC Forceps Major': 'Corpus callosum, forceps major', 'CC Forceps Minor': 'Corpus callosum, forceps minor',
               'CC Mid': 'Corpus callosum, body', 'CST': 'Corticospinal tract'}
FAMILY = {'Morphometry (Z-score)': 'Lobar composite z-score', 'Morphometry (Asymmetry)': 'Asymmetry index',
          'Tractometry (FA)': 'Tract-level FA', 'Tractometry (MD)': 'Tract-level MD'}
DOMAIN = {'Whole-brain morphometry (82 ROIs)': 'Morphometry, 82 regions',
          'Tractometry, tract-level (18 series)': 'Tractometry, 18 tract-level values',
          'Tractometry, node-level (3,000 values)': 'Tractometry, 3,000 node-level values'}
GROUP = {'Functional Seizures': 'Functional seizures', 'Drug-Resistant': 'Drug-resistant',
         'Drug-Responsive': 'Drug-responsive', 'Non-Lesional Epilepsy': 'Non-lesional epilepsy', 'FS': 'Functional seizures'}
COMP = {'Epilepsy vs. Functional Seizures': 'Epilepsy vs. functional seizures',
        'Focal vs. Generalized': 'Focal vs. generalized epilepsy',
        'Drug-Resistant vs. Drug-Responsive': 'Drug-resistant vs. drug-responsive epilepsy'}

def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def ci_text(s):  # '0.569 (0.482-0.654)' -> '0.57 (0.48-0.65)'
    auc, lo, hi = [float(x) for x in s.replace('(', '').replace(')', '').replace('-', ' ').split()]
    return f'{auc:.2f} ({lo:.2f}-{hi:.2f})'

rows = []
for comp in COMP:
    m = main[main['Comparison'] == comp]
    assert len(m) == 32, (comp, len(m))
    for dom, fam in FAMILY.items():
        for _, r in m[m['Domain'] == dom].iterrows():
            rows.append({'Comparison': COMP[comp], 'Measure': fam,
                         'Feature': TRACT_NAMES.get(r['Region/Tract'], r['Region/Tract'].capitalize()),
                         'AUC (95% CI)': ci_text(r['AUC (95% CI)']), 'Higher in': GROUP.get(r['Higher In'], r['Higher In']),
                         'p': fmt_p(r['p']), 'q': fmt_p(r['q'])})
    b = burden[burden['Comparison'] == comp]
    assert len(b) == 6, (comp, len(b))
    for _, r in b.iterrows():
        rows.append({'Comparison': COMP[comp], 'Measure': 'Whole-brain burden',
                     'Feature': f"{DOMAIN[r['Domain']]}: {'number with |z|>2' if r['Measure'].startswith('Regions') else 'mean |z|'}",
                     'AUC (95% CI)': ci_text(r['AUC (95% CI)']), 'Higher in': GROUP.get(r['Higher In'], r['Higher In']),
                     'p': fmt_p(r['p']), 'q': fmt_p(r['q'])})
    h = hip[hip['Comparison'] == comp]
    assert len(h) == 2, (comp, len(h))
    for _, r in h.iterrows():
        rows.append({'Comparison': COMP[comp], 'Measure': 'Hippocampal volume',
                     'Feature': 'Asymmetry |z_L - z_R|' if r['Measure'].startswith('|') else 'Mean of left and right z',
                     'AUC (95% CI)': ci_text(r['AUC (95% CI)']), 'Higher in': GROUP.get(r['Higher In'], r['Higher In']),
                     'p': fmt_p(r['p']), 'q': fmt_p(r['q'])})

# Additional analyses
for _, r in hip[hip['Comparison'] == 'MTS vs. Non-Lesional Epilepsy'].iterrows():
    rows.append({'Comparison': 'Unilateral MTS vs. non-lesional epilepsy (positive control)', 'Measure': 'Hippocampal volume',
                 'Feature': 'Asymmetry |z_L - z_R|' if r['Measure'].startswith('|') else 'Mean of left and right z',
                 'AUC (95% CI)': ci_text(r['AUC (95% CI)']), 'Higher in': GROUP.get(r['Higher In'], r['Higher In']),
                 'p': fmt_p(r['p']), 'q': fmt_p(r['q'])})
for _, r in nl.iterrows():
    rows.append({'Comparison': 'Non-lesional epilepsy vs. functional seizures', 'Measure': 'Whole-brain burden',
                 'Feature': f"{DOMAIN[r['Domain']]}: {'number with |z|>2' if r['Measure'].startswith('Regions') else 'mean |z|'}",
                 'AUC (95% CI)': ci_text(r['AUC (95% CI)']), 'Higher in': GROUP.get(r['Higher In'], r['Higher In']),
                 'p': fmt_p(r['p']), 'q': fmt_p(r['q'])})

table = pd.DataFrame(rows)
table.to_csv(f'{out_dir}/table_supp_discrimination_all.csv', index=False)
print(len(table), 'rows'); print(table.groupby('Comparison', sort=False).size().to_string())
pd.set_option('display.max_rows', 200); pd.set_option('display.width', 200)
table

128 rows
Comparison
Epilepsy vs. functional seizures                               40
Focal vs. generalized epilepsy                                 40
Drug-resistant vs. drug-responsive epilepsy                    40
Unilateral MTS vs. non-lesional epilepsy (positive control)     2
Non-lesional epilepsy vs. functional seizures                   6


,Comparison,Measure,Feature,AUC (95% CI),Higher in,p,q
0,Epilepsy vs. functional seizures,Lobar composite z-score,Frontal,0.57 (0.48-0.65),Epilepsy,0.200,0.698
1,Epilepsy vs. functional seizures,Lobar composite z-score,Temporal,0.55 (0.46-0.65),Epilepsy,0.316,0.737
2,Epilepsy vs. functional seizures,Lobar composite z-score,Parietal,0.59 (0.50-0.69),Functional seizures,0.081,0.570
3,Epilepsy vs. functional seizures,Lobar composite z-score,Occipital,0.53 (0.43-0.63),Epilepsy,0.622,0.845
4,Epilepsy vs. functional seizures,Lobar composite z-score,Insula,0.53 (0.43-0.63),Functional seizures,0.594,0.845
5,Epilepsy vs. functional seizures,Lobar composite z-score,Thalamus,0.50 (0.40-0.60),Epilepsy,0.989,0.989
6,Epilepsy vs. functional seizures,Lobar composite z-score,Basal ganglia,0.52 (0.43-0.61),Functional seizures,0.724,0.845
7,Epilepsy vs. functional seizures,Asymmetry index,Frontal,0.57 (0.47-0.68),Functional seizures,0.175,0.307
8,Epilepsy vs. functional seizures,Asymmetry index,Temporal,0.60 (0.51-0.69),Epilepsy,0.056,0.197
9,Epilepsy vs. functional seizures,Asymmetry index,Parietal,0.55 (0.45-0.65),Epilepsy,0.376,0.439
